
Часть 1. Эмбеддинги на основе Word2Vec

Используя ранее обученную модель Word2Vec, решите задачу классификации текста(ов):

1. Получите векторные представления слов из обученной модели.
2. Постройте эмбеддинг для каждого текста (например, как среднее векторных представлений входящих в него слов).
3. Обучите классификатор на полученных признаках и оцените качество с помощью метрики F1-score (macro).

Часть 2. Эмбеддинги из BERT

Получите векторные представления текстов с помощью предобученной модели BERT двумя способами:

• 2.1. В качестве вектора текста используйте эмбеддинг токена [CLS].
• 2.2. Усредните эмбеддинги всех токенов текста (аналогично подходу из части 1).

Используя полученные представления, обучите классификатор и сравните результаты с частью 1. Попробуйте улучшить ранее полученные результаты классификации за счёт представления текстов через BERT-эмбеддинги. Сравните все подходы (Word2Vec, BERT-CLS, BERT-mean) по F1-score и сделайте выводы.

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 37.2 MB/s eta 0:00:00


In [2]:
import torch.nn as nn
import nltk
import torch
from nltk.stem import WordNetLemmatizer
from torch.utils.data import Dataset, DataLoader
from nltk.corpus import stopwords, wordnet
import spacy
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
import gensim.downloader as api
from datasets import load_dataset
from tqdm import tqdm
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from collections import Counter

In [3]:
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
class Preprocessor:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()
    def clean(self, text):
        if not isinstance(text, str):
            text = str(text)
        text = text.lower()
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        return text
    def preprocess_single(self, text):
        if isinstance(text, list):
            text = text[0] if text else ""
        elif not isinstance(text, str):
            text = str(text)
        text = self.clean(text)
        try:
            tokens = nltk.word_tokenize(text)
        except:
            tokens = text.split()
        processed_tokens = []
        for token in tokens:
            if token and token not in self.stop_words:
                lemma = self.lemmatizer.lemmatize(token)
                processed_tokens.append(lemma)
        return ' '.join(processed_tokens)
    def preprocess(self, texts):
        if isinstance(texts, list):
            return [self.preprocess_single(t) for t in texts]
        else:
            return self.preprocess_single(texts)

In [5]:
class EmotionDataset(Dataset):
  def __init__(self, texts, labels, preprocessor,word2vec_model, max_length=100, embedding_dim=128):
    self.texts = texts
    self.labels = labels
    self.max_length = max_length
    self.embedding_dim = embedding_dim
    self.preprocessor = preprocessor
    self.word2vec_model = word2vec_model
  def __len__(self):
    return len(self.texts)
  def text_to_sequence(self, tokens):
    tokens = self.preprocessor.preprocess(tokens)
    tokens = tokens.split()
    sequence = []
    for token in tokens:
      if token in self.word2vec_model.wv:
        sequence.append(self.word2vec_model.wv[token].astype(np.float32))
      else:
        sequence.append(np.zeros(self.embedding_dim, dtype=np.float32))
    if len(sequence) > self.max_length:
      sequence = sequence[:self.max_length]
    else:
      padding = [np.zeros(self.embedding_dim)] * (self.max_length - len(sequence))
      sequence.extend(padding)
    return np.array(sequence, dtype=np.float32)
  def __getitem__(self, index):
    text = self.texts[index]
    label = self.labels[index]
    seq = self.text_to_sequence(text)
    return torch.tensor(seq, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [6]:
class MeanDataset(Dataset):
  def __init__(self, texts, labels, preprocessor,word2vec_model, max_length=100, embedding_dim=128):
    self.texts = texts
    self.labels = labels
    self.max_length = max_length
    self.embedding_dim = embedding_dim
    self.preprocessor = preprocessor
    self.word2vec_model = word2vec_model
  def __len__(self):
    return len(self.texts)
  def text_to_sequence(self, tokens):
    tokens = self.preprocessor.preprocess(tokens)
    tokens = tokens.split()
    sequence = []
    for token in tokens:
      if token in self.word2vec_model.wv:
        sequence.append(self.word2vec_model.wv[token].astype(np.float32))
      else:
        sequence.append(np.zeros(self.embedding_dim, dtype=np.float32))
    if len(sequence) > self.max_length:
      sequence = sequence[:self.max_length]
    else:
      padding = [np.zeros(self.embedding_dim)] * (self.max_length - len(sequence))
      sequence.extend(padding)
    sequence = np.mean(sequence, axis=0)
    return np.array(sequence, dtype=np.float32)
  def __getitem__(self, index):
    text = self.texts[index]
    label = self.labels[index]
    seq = self.text_to_sequence(text)
    return torch.tensor(seq, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [7]:
def train_epoch(model, criterion, optimizer, scheduler,train_loader,device):
  model.train()
  epoch_loss = 0
  train_pbar = tqdm(train_loader, desc='Train')
  for idx, (text, label) in enumerate(train_pbar):
    text, label = text.to(device), label.to(device)
    optimizer.zero_grad()
    output = model(text)
    loss = criterion(output, label)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    epoch_loss += loss.item()
    probs = torch.softmax(output, dim=1)
    _,preds = torch.max(probs, dim=1)
    preds, label = preds.cpu().numpy(), label.cpu().numpy()
    accuracy = accuracy_score(preds, label)
    precision = precision_score(preds, label, average='weighted', zero_division=0)
    recall = recall_score(preds, label, average='weighted', zero_division=0)
    f1_score_micro = f1_score(preds,label,average='micro', zero_division=0)
    f1_score_macro = f1_score(preds,label,average='macro', zero_division=0)
    f1_score_weight = f1_score(preds,label,average='weighted', zero_division=0)
    train_pbar.set_postfix({
        'loss': f'{epoch_loss/(idx+1):.4f}',
        'acc': f'{accuracy:.4f}',
        'precision': f'{precision:.4f}',
        'recall': f'{recall:.4f}',
        'f1_micro': f'{f1_score_micro:.4f}',
        'f1_macro': f'{f1_score_macro:.4f}',
        'f1_weight': f'{f1_score_weight:.4f}'
    })
  train_pbar.close()
  avg_loss = epoch_loss / len(train_loader)
  return avg_loss


In [8]:
def evaluate_epoch(model, criterion, val_loader, device):
    model.eval()
    epoch_loss = 0
    all_preds = []
    all_labels = []
    val_pbar = tqdm(val_loader, desc='Val')
    with torch.no_grad():
        for text, label in val_pbar:
            text, label = text.to(device), label.to(device)
            output = model(text)
            loss = criterion(output, label)
            epoch_loss += loss.item()
            _, preds = torch.max(output, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.cpu().numpy())
            current_acc = accuracy_score(all_labels, all_preds)
            val_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{current_acc:.4f}'})
    val_pbar.close()
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_micro = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_weight = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    metrics = {
        'acc': f'{accuracy:.4f}',
        'precision': f'{precision:.4f}',
        'recall': f'{recall:.4f}',
        'f1_micro': f'{f1_micro:.4f}',
        'f1_macro': f'{f1_macro:.4f}',
        'f1_weight': f'{f1_weight:.4f}'
    }

    avg_loss = epoch_loss / len(val_loader)
    return metrics, avg_loss

In [9]:
def early_stopping(cur_metric, best_metric, patience_count, patience, min_d):
    if cur_metric > best_metric + min_d:
        better = True
    else:
        better = False
    if better:
        patience_count = 0
    else:
        patience_count += 1
    stop = patience_count >= patience
    return stop, patience_count

In [10]:
class BiLSTMWithAttention(nn.Module):
  def __init__(self,embedding_dim,hidden_dim,num_classes,p):
    super().__init__()
    self.lstm = nn.LSTM(
        embedding_dim, hidden_dim, 2,
        batch_first=True, bidirectional=True, dropout=p
    )
    self.attention = nn.Sequential(
        nn.Linear(hidden_dim * 2, hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim,1)
    )
    self.fc = nn.Linear(hidden_dim * 2, num_classes)
    self.dropout = nn.Dropout(p)
  def forward(self,x):
    out, _ = self.lstm(x)
    attention_weights = self.attention(out)
    attention_weights = F.softmax(attention_weights,dim=1)
    weighted = (out * attention_weights).sum(dim=1)
    weighted = self.dropout(weighted)
    return self.fc(weighted)

In [11]:
class Classifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.5)
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

In [12]:
import torch.nn.functional as F

ATTENTION BI LSTM + WORD2VEC - полные эмбеддинги

In [13]:
if __name__ == '__main__':
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  dt = load_dataset('emotion')
  train_texts = list(dt['train']['text'])
  train_labels = list(dt['train']['label'])
  test_texts = list(dt['test']['text'])
  test_labels = list(dt['test']['label'])
  X_train, X_val, y_train, y_val = train_test_split(train_texts, train_labels, test_size=0.2)
  encoder = LabelEncoder()
  y_train = encoder.fit_transform(y_train)
  y_val = encoder.transform(y_val)
  from gensim.models import Word2Vec
  texts = [text.split() for text in X_train]
  word2vec_model = Word2Vec(
      sentences=texts,
      vector_size=128,
      window=5,
      min_count=3,
      workers=4,
      epochs=20,
      sg=1
  )
  preprocessor = Preprocessor()
  train_dataset = EmotionDataset(X_train,y_train, preprocessor, word2vec_model)
  test_dataset = EmotionDataset(X_val,y_val, preprocessor, word2vec_model)
  train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=64,shuffle=False)
  num_classes = len(set(train_labels))

  model1 = BiLSTMWithAttention(128,64, num_classes, p=0.3).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.AdamW(
    model1.parameters(),
    lr=0.0002,
    weight_decay=0.0005)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
  best = 0
  patience_count = 0
  for epoch in range(50):
    print(epoch)
    avg_loss = train_epoch(model1,criterion,optimizer,scheduler,train_loader,device)
    metrics, val_loss = evaluate_epoch(model1,criterion,test_loader, device)
    scheduler.step(val_loss)
    print(metrics)
    if best <= float(metrics['f1_micro']):
      best = float(metrics['f1_micro'])
    stop, patience_count = early_stopping(
                float(metrics['f1_micro']),
                best_metric=best,
                patience_count=patience_count,
                patience=50,
                min_d=0.001
        )
    if stop:
      print(f"early stopping")
      break
  print(best)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

{'acc': '0.9041', 'precision': '0.9039', 'recall': '0.9041', 'f1_micro': '0.9041', 'f1_macro': '0.8703', 'f1_weight': '0.9038'}
48


Val: 100%|██████████| 50/50 [00:01<00:00, 40.40it/s, loss=0.0794, acc=0.9044]


{'acc': '0.9044', 'precision': '0.9042', 'recall': '0.9044', 'f1_micro': '0.9044', 'f1_macro': '0.8702', 'f1_weight': '0.9042'}
49


Val: 100%|██████████| 50/50 [00:01<00:00, 29.72it/s, loss=0.0791, acc=0.9034]


{'acc': '0.9034', 'precision': '0.9032', 'recall': '0.9034', 'f1_micro': '0.9034', 'f1_macro': '0.8695', 'f1_weight': '0.9032'}
early stopping
0.9047


НЕЙРОННЫЙ КЛАССИФИКАТОР + WORD2VEC + УСРЕДНЕННЫЕ ЭМБЕДДИНГИ

In [14]:
if __name__ == '__main__':
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  dt = load_dataset('emotion')
  train_texts = list(dt['train']['text'])
  train_labels = list(dt['train']['label'])
  test_texts = list(dt['test']['text'])
  test_labels = list(dt['test']['label'])
  X_train, X_val, y_train, y_val = train_test_split(train_texts, train_labels, test_size=0.2)
  encoder = LabelEncoder()
  y_train = encoder.fit_transform(y_train)
  y_val = encoder.transform(y_val)
  from gensim.models import Word2Vec
  texts = [text.split() for text in X_train]
  word2vec_model = Word2Vec(
      sentences=texts,
      vector_size=128,
      window=5,
      min_count=3,
      workers=4,
      epochs=20,
      sg=1
  )
  preprocessor = Preprocessor()
  train_dataset = MeanDataset(X_train,y_train, preprocessor, word2vec_model)
  test_dataset = MeanDataset(X_val,y_val, preprocessor, word2vec_model)
  train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=64,shuffle=False)
  num_classes = len(set(train_labels))

  model1 = Classifier(input_dim=128, num_classes=num_classes,
                      hidden_dims=[256,128]).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.AdamW(
    model1.parameters(),
    lr=0.0002,
    weight_decay=0.0005)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
  best = 0
  patience_count = 0
  for epoch in range(50):
    print(epoch)
    avg_loss = train_epoch(model1,criterion,optimizer,scheduler,train_loader,device)
    metrics, val_loss = evaluate_epoch(model1,criterion,test_loader, device)
    scheduler.step(val_loss)
    print(metrics)
    if best <= float(metrics['f1_micro']):
      best = float(metrics['f1_micro'])
    stop, patience_count = early_stopping(
                float(metrics['f1_micro']),
                best_metric=best,
                patience_count=patience_count,
                patience=50,
                min_d=0.001
        )
    if stop:
      print(f"early stopping")
      break
  print(best)

0


Val: 100%|██████████| 50/50 [00:01<00:00, 33.67it/s, loss=1.4899, acc=0.4841]


{'acc': '0.4841', 'precision': '0.4397', 'recall': '0.4841', 'f1_micro': '0.4841', 'f1_macro': '0.2138', 'f1_weight': '0.3858'}
1


Val: 100%|██████████| 50/50 [00:01<00:00, 41.27it/s, loss=1.4137, acc=0.5031]


{'acc': '0.5031', 'precision': '0.4557', 'recall': '0.5031', 'f1_micro': '0.5031', 'f1_macro': '0.2651', 'f1_weight': '0.4271'}
2


...

{'acc': '0.6372', 'precision': '0.6310', 'recall': '0.6372', 'f1_micro': '0.6372', 'f1_macro': '0.5639', 'f1_weight': '0.6301'}
48


Val: 100%|██████████| 50/50 [00:01<00:00, 34.22it/s, loss=1.0641, acc=0.6391]


{'acc': '0.6391', 'precision': '0.6308', 'recall': '0.6391', 'f1_micro': '0.6391', 'f1_macro': '0.5703', 'f1_weight': '0.6318'}
49


Val: 100%|██████████| 50/50 [00:01<00:00, 40.83it/s, loss=1.0610, acc=0.6388]


{'acc': '0.6388', 'precision': '0.6305', 'recall': '0.6388', 'f1_micro': '0.6388', 'f1_macro': '0.5658', 'f1_weight': '0.6301'}
early stopping
0.6391


In [15]:
from transformers import BertTokenizer, BertModel

In [16]:
class BertDataset(Dataset):
  def __init__(self,texts,labels, bert_model, tokenizer, device, max_length=128):
    self.texts = texts
    self.labels = labels
    self.max_length = max_length
    self.device = device
    self.embeddings = self._compute_embeddings(texts,bert_model,tokenizer)
  def _compute_embeddings(self, texts, bert_model,tokenizer):
    bert_model.eval()
    embeds = []
    for text in tqdm(texts,desc='embeds'):
      encoded = tokenizer(
          text,
          padding='max_length',
          truncation=True,
          max_length=self.max_length,
          return_tensors='pt'
      )
      input_ids = encoded['input_ids'].to(self.device)
      attention_mask = encoded['attention_mask'].to(self.device)
      with torch.no_grad():
        out = bert_model(input_ids, attention_mask=attention_mask)
        token_embeddings = out.last_hidden_state
        embeds.append(token_embeddings.cpu().numpy())
    return np.vstack(embeds)
  def __len__(self):
    return len(self.texts)
  def __getitem__(self,idx):
    embeds = torch.tensor(self.embeddings[idx],dtype=torch.float32)
    label = torch.tensor(self.labels[idx],dtype=torch.long)
    return embeds, label


In [17]:
class BiLSTMWithMultiAttention(nn.Module):
  def __init__(self,embedding_dim,hidden_dim,num_classes,p):
    super().__init__()
    self.lstm = nn.LSTM(
        embedding_dim, hidden_dim, 2,
        batch_first=True, bidirectional=True, dropout=p
    )
    self.attn = nn.MultiheadAttention(
        embed_dim=hidden_dim * 2,
        num_heads = 4,
        dropout=p,
        batch_first=True
    )
    self.classifier = nn.Sequential(
        nn.Linear(hidden_dim * 2, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(hidden_dim, num_classes)
    )
    self.dropout = nn.Dropout(p)
  def forward(self,x):
    out, _ = self.lstm(x)
    attn_out, attention_weights = self.attn(out,out,out)
    avg_pool = attn_out.mean(dim=1)
    max_pool = attn_out.max(dim=1)[0]
    combined = torch.cat([avg_pool, max_pool],dim=1)
    return self.classifier(combined)

ATTENTION BI LSTM + BERT + ПОЛНЫЕ ЭМБЕДДИНГИ

In [18]:
if __name__ == '__main__':
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  dt = load_dataset('emotion')
  train_texts = list(dt['train']['text'])
  train_labels = list(dt['train']['label'])
  test_texts = list(dt['test']['text'])
  test_labels = list(dt['test']['label'])
  X_train, X_val, y_train, y_val = train_test_split(train_texts, train_labels, test_size=0.2)
  encoder = LabelEncoder()
  y_train = encoder.fit_transform(y_train)
  y_val = encoder.transform(y_val)
  tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
  train_dataset = BertDataset(X_train, y_train, bert_model, tokenizer, device)
  test_dataset = BertDataset(X_val,y_val, bert_model,tokenizer,device)
  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=32,shuffle=False)
  num_classes = len(set(train_labels))

  model1 = BiLSTMWithAttention(bert_model.config.hidden_size,128, num_classes, p=0.6).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.AdamW(
    model1.parameters(),
    lr=0.002,
    weight_decay=0.001)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
  best = 0
  patience_count = 0
  for epoch in range(30):
    print(epoch)
    avg_loss = train_epoch(model1,criterion,optimizer,scheduler,train_loader,device)
    metrics, val_loss = evaluate_epoch(model1,criterion,test_loader, device)
    scheduler.step(val_loss)
    print(metrics)
    if best <= float(metrics['f1_micro']):
      best = float(metrics['f1_micro'])
    stop, patience_count = early_stopping(
                float(metrics['f1_micro']),
                best_metric=best,
                patience_count=patience_count,
                patience=50,
                min_d=0.001
        )
    if stop:
      print(f"early stopping")
      break
  print(best)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

{'acc': '0.9006', 'precision': '0.9006', 'recall': '0.9006', 'f1_micro': '0.9006', 'f1_macro': '0.8611', 'f1_weight': '0.9004'}
28


Val: 100%|██████████| 100/100 [00:01<00:00, 58.47it/s, loss=0.0663, acc=0.9006]


{'acc': '0.9006', 'precision': '0.9006', 'recall': '0.9006', 'f1_micro': '0.9006', 'f1_macro': '0.8611', 'f1_weight': '0.9004'}
29


Val: 100%|██████████| 100/100 [00:01<00:00, 52.15it/s, loss=0.0663, acc=0.9006]


{'acc': '0.9006', 'precision': '0.9006', 'recall': '0.9006', 'f1_micro': '0.9006', 'f1_macro': '0.8611', 'f1_weight': '0.9004'}
0.9053


In [19]:
class BertEmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels, model, tokenizer, device, choose='cls'):
      if choose == 'cls':
        self.embeddings = torch.tensor(self._get_embeds_cls(embeddings,model,tokenizer,device), dtype=torch.float32)
      else:
        self.embeddings = torch.tensor(self._get_embeds_mean(embeddings,model,tokenizer,device), dtype=torch.float32)
      self.labels = torch.tensor(labels, dtype=torch.long)
    def _get_embeds_cls(self, texts,model,tokenizer,device, batch_size=32):
      model.eval()
      embeddings = []
      for i in tqdm(range(0, len(texts), batch_size), desc="CLS embeddings"):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            embeddings.append(cls_embeddings.cpu().numpy())
      return np.vstack(embeddings)
    def _get_embeds_mean(self,texts, model, tokenizer, device,batch_size=32):
      model.eval()
      embeddings = []
      for i in tqdm(range(0, len(texts), batch_size), desc="Mean embeddings"):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)
            token_embeddings = outputs.last_hidden_state
            attention_mask_expanded = attention_mask.unsqueeze(-1).float()
            sum_embeddings = (token_embeddings * attention_mask_expanded).sum(dim=1)
            mean_embeddings = sum_embeddings / attention_mask_expanded.sum(dim=1)
            embeddings.append(mean_embeddings.cpu().numpy())
      return np.vstack(embeddings)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


ПРОСТОЙ НЕЙРОННЫЙ КЛАССИФИКАТОР + BERT ЭМБЕДИГИ ЧЕРЕЗ CLS ТОКЕН

In [20]:
if __name__ == '__main__':
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  dt = load_dataset('emotion')
  train_texts = list(dt['train']['text'])
  train_labels = list(dt['train']['label'])
  test_texts = list(dt['test']['text'])
  test_labels = list(dt['test']['label'])
  X_train, X_val, y_train, y_val = train_test_split(train_texts, train_labels, test_size=0.2)
  encoder = LabelEncoder()
  y_train = encoder.fit_transform(y_train)
  y_val = encoder.transform(y_val)
  tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
  train_dataset = BertEmbeddingDataset(X_train, y_train, bert_model, tokenizer, device)
  test_dataset = BertEmbeddingDataset(X_val,y_val, bert_model,tokenizer,device)
  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=32,shuffle=False)
  num_classes = len(set(train_labels))

  model1 = Classifier(input_dim=bert_model.config.hidden_size, num_classes=num_classes,
                      hidden_dims=[256,128]).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.AdamW(
    model1.parameters(),
    lr=0.0002,
    weight_decay=0.001)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
  best = 0
  patience_count = 0
  for epoch in range(50):
    print(epoch)
    avg_loss = train_epoch(model1,criterion,optimizer,scheduler,train_loader,device)
    metrics, val_loss = evaluate_epoch(model1,criterion,test_loader, device)
    scheduler.step(val_loss)
    print(metrics)
    if best <= float(metrics['f1_micro']):
      best = float(metrics['f1_micro'])
    stop, patience_count = early_stopping(
                float(metrics['f1_micro']),
                best_metric=best,
                patience_count=patience_count,
                patience=50,
                min_d=0.001
        )
    if stop:
      print(f"early stopping")
      break
  print(best)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
CLS embeddings: 100%|██████████| 100/100 [00:10<00:00,  9.33it/s]


0


Val: 100%|██████████| 100/100 [00:00<00:00, 185.35it/s, loss=1.3057, acc=0.5363]


{'acc': '0.5363', 'precision': '0.4974', 'recall': '0.5363', 'f1_micro': '0.5363', 'f1_macro': '0.3009', 'f1_weight': '0.4661'}
1


...

{'acc': '0.6050', 'precision': '0.5952', 'recall': '0.6050', 'f1_micro': '0.6050', 'f1_macro': '0.5032', 'f1_weight': '0.5976'}
48


Val: 100%|██████████| 100/100 [00:00<00:00, 169.67it/s, loss=1.1632, acc=0.6078]


{'acc': '0.6078', 'precision': '0.5972', 'recall': '0.6078', 'f1_micro': '0.6078', 'f1_macro': '0.5059', 'f1_weight': '0.6000'}
49


Val: 100%|██████████| 100/100 [00:00<00:00, 213.26it/s, loss=1.1285, acc=0.6091]


{'acc': '0.6091', 'precision': '0.6007', 'recall': '0.6091', 'f1_micro': '0.6091', 'f1_macro': '0.5103', 'f1_weight': '0.6017'}
early stopping
0.6119


ПРОСТОЙ НЕЙРОННЫЙ КЛАССИФИКАТОР + BERT ЭМБЕДДИНГИ УСРЕДНЕННЫЕ

In [21]:
if __name__ == '__main__':
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  dt = load_dataset('emotion')
  train_texts = list(dt['train']['text'])
  train_labels = list(dt['train']['label'])
  test_texts = list(dt['test']['text'])
  test_labels = list(dt['test']['label'])
  X_train, X_val, y_train, y_val = train_test_split(train_texts, train_labels, test_size=0.2)
  encoder = LabelEncoder()
  y_train = encoder.fit_transform(y_train)
  y_val = encoder.transform(y_val)
  tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
  train_dataset = BertEmbeddingDataset(X_train, y_train, bert_model, tokenizer, device, choose='d')
  test_dataset = BertEmbeddingDataset(X_val,y_val, bert_model,tokenizer,device,choose='g')
  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=32,shuffle=False)
  num_classes = len(set(train_labels))

  model1 = Classifier(input_dim=bert_model.config.hidden_size, num_classes=num_classes,
                      hidden_dims=[256,128]).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.AdamW(
    model1.parameters(),
    lr=0.0002,
    weight_decay=0.0005)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
  best = 0
  patience_count = 0
  for epoch in range(50):
    print(epoch)
    avg_loss = train_epoch(model1,criterion,optimizer,scheduler,train_loader,device)
    metrics, val_loss = evaluate_epoch(model1,criterion,test_loader, device)
    scheduler.step(val_loss)
    print(metrics)
    if best <= float(metrics['f1_micro']):
      best = float(metrics['f1_micro'])
    stop, patience_count = early_stopping(
                float(metrics['f1_micro']),
                best_metric=best,
                patience_count=patience_count,
                patience=50,
                min_d=0.001
        )
    if stop:
      print(f"early stopping")
      break
  print(best)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Mean embeddings: 100%|██████████| 100/100 [00:10<00:00,  9.10it/s]


0


Val: 100%|██████████| 100/100 [00:00<00:00, 189.75it/s, loss=1.0662, acc=0.5841]


{'acc': '0.5841', 'precision': '0.5575', 'recall': '0.5841', 'f1_micro': '0.5841', 'f1_macro': '0.3894', 'f1_weight': '0.5402'}
1


...

{'acc': '0.6631', 'precision': '0.6577', 'recall': '0.6631', 'f1_micro': '0.6631', 'f1_macro': '0.5802', 'f1_weight': '0.6564'}
48


Val: 100%|██████████| 100/100 [00:00<00:00, 210.19it/s, loss=0.7813, acc=0.6628]


{'acc': '0.6628', 'precision': '0.6594', 'recall': '0.6628', 'f1_micro': '0.6628', 'f1_macro': '0.5814', 'f1_weight': '0.6560'}
49


Val: 100%|██████████| 100/100 [00:00<00:00, 222.54it/s, loss=0.7760, acc=0.6634]


{'acc': '0.6634', 'precision': '0.6572', 'recall': '0.6634', 'f1_micro': '0.6634', 'f1_macro': '0.5767', 'f1_weight': '0.6569'}
early stopping
0.6663


 Добавляем классификаторы:

Multinomial Naive Bayes
Logistic Regression
Linear SVM
SGDClassifier
+ опционально kNN

EMOTION

W2V

In [22]:
import numpy as np
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

In [23]:
def extract_embeddings_from_dataset(dataset):
    embeddings = []
    labels = []
    for i in range(len(dataset)):
        emb, label = dataset[i]
        embeddings.append(emb.numpy())
        labels.append(label.numpy())
    return np.array(embeddings), np.array(labels)

In [26]:
if __name__ == '__main__':
  train_dataset_mean = MeanDataset(X_train, y_train, preprocessor, word2vec_model)
  val_dataset_mean = MeanDataset(X_val, y_val, preprocessor, word2vec_model)
  test_dataset_mean = MeanDataset(test_texts, test_labels, preprocessor, word2vec_model)
  X_train_w2v, y_train_w2v = extract_embeddings_from_dataset(train_dataset_mean)
  X_val_w2v, y_val_w2v = extract_embeddings_from_dataset(val_dataset_mean)
  X_test_w2v, y_test_w2v = extract_embeddings_from_dataset(test_dataset_mean)
  scaler = StandardScaler()
  X_train_w2v = scaler.fit_transform(X_train_w2v)
  X_val_w2v = scaler.transform(X_val_w2v)
  X_test_w2v = scaler.transform(X_test_w2v)
  classifiers = {
    'MultinomialNB': MultinomialNB(),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'LinearSVM': LinearSVC(max_iter=2000, random_state=42, dual='auto'),
    'SGDClassifier': SGDClassifier(loss='log_loss', max_iter=1000, random_state=42)
  }
  results_w2v = {}

  for name, clf in classifiers.items():
    if name == 'MultinomialNB':
        X_train_clf = np.abs(X_train_w2v)
        X_val_clf = np.abs(X_val_w2v)
        X_test_clf = np.abs(X_test_w2v)
    else:
        X_train_clf = X_train_w2v
        X_val_clf = X_val_w2v
        X_test_clf = X_test_w2v

    clf.fit(X_train_clf, y_train_w2v)
    y_pred_val = clf.predict(X_val_clf)
    y_pred_test = clf.predict(X_test_clf)

    f1_val = f1_score(y_val_w2v, y_pred_val, average='micro')
    f1_test = f1_score(y_test_w2v, y_pred_test, average='micro')

    results_w2v[name] = {'val_f1': f1_val, 'test_f1': f1_test}
    print(name)
    print(f"Validation F1: {f1_val:.4f}")
    print(f"Test F1: {f1_test:.4f}")

MultinomialNB
Validation F1: 0.3563
Test F1: 0.3825
LogisticRegression
Validation F1: 0.5828
Test F1: 0.5950
LinearSVM
Validation F1: 0.5681
Test F1: 0.5825
SGDClassifier
Validation F1: 0.5697
Test F1: 0.5705


BERT[CLS]

In [29]:
results = {}

In [30]:
if __name__ == '__main__':
  train_bert_cls = BertEmbeddingDataset(X_train, y_train, bert_model, tokenizer, device, choose='cls')
  val_bert_cls = BertEmbeddingDataset(X_val, y_val, bert_model, tokenizer, device, choose='cls')
  test_bert_cls = BertEmbeddingDataset(test_texts, test_labels, bert_model, tokenizer, device, choose='cls')
  X_train_cls, y_train_cls = extract_embeddings_from_dataset(train_bert_cls)
  X_val_cls, y_val_cls = extract_embeddings_from_dataset(val_bert_cls)
  X_test_cls, y_test_cls = extract_embeddings_from_dataset(test_bert_cls)
  scaler_cls = StandardScaler()
  X_train_cls = scaler_cls.fit_transform(X_train_cls)
  X_val_cls = scaler_cls.transform(X_val_cls)
  X_test_cls = scaler_cls.transform(X_test_cls)
  results['BERT_CLS'] = {}

  for name, clf in classifiers.items():
    print(f"\nTraining {name} on BERT [CLS]...")

    if name == 'MultinomialNB':
        X_train_clf = np.abs(X_train_cls)
        X_test_clf = np.abs(X_test_cls)
    else:
        X_train_clf = X_train_cls
        X_test_clf = X_test_cls

    clf.fit(X_train_clf, y_train_cls)
    y_pred = clf.predict(X_test_clf)
    f1 = f1_score(y_test_cls, y_pred, average='micro')
    print(name)
    results['BERT_CLS'][name] = f1
    print(f"  Test F1: {f1:.4f}")

CLS embeddings: 100%|██████████| 63/63 [00:06<00:00,  9.46it/s]



Training MultinomialNB on BERT [CLS]...
MultinomialNB
  Test F1: 0.3900

Training LogisticRegression on BERT [CLS]...
LogisticRegression
  Test F1: 0.5665

Training LinearSVM on BERT [CLS]...
LinearSVM
  Test F1: 0.5800

Training SGDClassifier on BERT [CLS]...
SGDClassifier
  Test F1: 0.5510


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


BERT[MEAN]

In [32]:
if __name__ == '__main__':
  train_bert_mean = BertEmbeddingDataset(X_train, y_train, bert_model, tokenizer, device, choose='mean')
  val_bert_mean = BertEmbeddingDataset(X_val, y_val, bert_model, tokenizer, device, choose='mean')
  test_bert_mean = BertEmbeddingDataset(test_texts, test_labels, bert_model, tokenizer, device, choose='mean')
  X_train_mean, y_train_mean = extract_embeddings_from_dataset(train_bert_mean)
  X_val_mean, y_val_mean = extract_embeddings_from_dataset(val_bert_mean)
  X_test_mean, y_test_mean = extract_embeddings_from_dataset(test_bert_mean)

  scaler_mean = StandardScaler()
  X_train_mean = scaler_mean.fit_transform(X_train_mean)
  X_val_mean = scaler_mean.transform(X_val_mean)
  X_test_mean = scaler_mean.transform(X_test_mean)
  results['BERT_Mean'] = {}

  for name, clf in classifiers.items():
    print(f"\nTraining {name} on BERT Mean...")

    if name == 'MultinomialNB':
        X_train_clf = np.abs(X_train_mean)
        X_test_clf = np.abs(X_test_mean)
    else:
        X_train_clf = X_train_mean
        X_test_clf = X_test_mean

    clf.fit(X_train_clf, y_train_mean)
    y_pred = clf.predict(X_test_clf)
    f1 = f1_score(y_test_mean, y_pred, average='micro')
    print(name)
    results['BERT_Mean'][name] = f1
    print(f"  Test F1: {f1:.4f}")

Mean embeddings: 100%|██████████| 63/63 [00:06<00:00,  9.55it/s]



Training MultinomialNB on BERT Mean...
MultinomialNB
  Test F1: 0.4270

Training LogisticRegression on BERT Mean...
LogisticRegression
  Test F1: 0.6100

Training LinearSVM on BERT Mean...
LinearSVM
  Test F1: 0.6345

Training SGDClassifier on BERT Mean...
SGDClassifier
  Test F1: 0.6035


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
